<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/week3/lesson10/Lesson10_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 10 — Exercises**
## The Chain Rule and Backpropagation: How a Change Travels Through a Network

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 3 — Calculus I\
**Lesson:** Lesson 10

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | The chain rule by hand | 10 |
| 2 | A computation graph: the Poisson neuron | 15 |
| 3 | Extending an automatic-differentiation engine | 15 |
| 4 | Backpropagation through an affine layer | 15 |
| 5 | Vanishing and exploding gradients through time | 15 |
| 6 | Input sensitivity of a network | 15 |
| 7 | Real data: backpropagation for a deeper decoder | 15 |
| **Total** | | **100** |
| Bonus | Forward-mode differentiation | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(7)   # exercise notebook uses its own fixed seed
PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def central_diff(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

print("Setup complete.")

---
## Exercise 1 — The Chain Rule by Hand (10 points)

**1.1 (4 points, written response, no code).** Differentiate, naming the inner function $u$ each time:

(a) $V(t) = V_0\,e^{-(t/\tau)^2}$ (a Gaussian-shaped voltage transient; $t$ in ms, $V$ in mV) — give the units of $dV/dt$;
(b) $r(c) = r_{\max}\,\tanh(g\,\ln(1 + c))$ (a log-compressed, saturating response to contrast $c$);
(c) $p(x) = \sigma\big(w_2\,\tanh(w_1x + b_1) + b_2\big)$ (a two-layer network with one unit per layer), with respect to $x$.

**Your answer:**

*(replace this line with your answer)*

In [ ]:
# 1.2 (3 points): Code the three functions and your three derivatives (as functions of t, c and x), and check each with
# central_diff at three points of your choice. Use V0 = 10, tau = 5; r_max = 40, g = 0.8; w1 = 1.5, b1 = -0.5, w2 = 2.0, b2 = 0.3.
# Store the largest relative differences in chain_err (dict with keys 'a', 'b', 'c').

V0, tau, r_max, g = 10.0, 5.0, 40.0, 0.8
w1, b1, w2, b2 = 1.5, -0.5, 2.0, 0.3

# YOUR CODE HERE
chain_err = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2
assert chain_err is not None and set(chain_err) == {'a', 'b', 'c'}, "chain_err needs keys 'a', 'b', 'c'."
assert all(v < 1e-6 for v in chain_err.values()), "At least one derivative disagrees with its central difference."
print("Self-check passed.")

**1.3 (3 points, written response).** In (c), at which inputs $x$ is $p'(x)$ close to zero even though $w_1$ and $w_2$ are not small? Explain with the factors of the chain.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 2 — A Computation Graph: the Poisson Neuron (15 points)

A standard model of spike counts (Lesson 18) gives a neuron an expected count $\lambda = e^{wx + b}$ in a time bin, and measures how badly it predicts an observed count $y$ by the **Poisson loss** (negative log-likelihood, up to a constant)

$$L = \lambda - y\ln\lambda .$$

The computation graph is $w, x, b \to z = wx + b \to \lambda = e^{z} \to L$.

**2.1 (5 points, written response).** Do the backward pass by hand: find $\partial L/\partial\lambda$, $\partial\lambda/\partial z$, and then $\partial L/\partial z$, $\partial L/\partial w$ and $\partial L/\partial b$. Simplify $\partial L/\partial z$ as far as possible and interpret it in words. (Hint: $\ln\lambda = z$, so you can also write $L = e^{z} - yz$.)

**Your answer:**

*(replace this line with your answer)*

In [ ]:
# 2.2 (6 points): Implement poisson_forward_backward(w, b, x, y) for ARRAYS x and y of n examples, with the loss
# averaged over examples: L = mean(lam - y * log(lam)). Return (L, dL_dw, dL_db). Then gradient-check it with central
# differences on the data below, storing the relative errors in pois_err = (err_w, err_b).

x2 = RNG.normal(size=200)
y2 = RNG.poisson(np.exp(0.8 * x2 + 0.5))                 # spike counts generated with w = 0.8, b = 0.5

def poisson_forward_backward(w, b, x, y):
    # YOUR CODE HERE
    raise NotImplementedError

pois_err = None

In [ ]:
# SELF-CHECK 2.2
assert pois_err is not None and len(pois_err) == 2, "pois_err should hold two relative errors."
assert max(pois_err) < 1e-6, "The analytic gradient disagrees with the central difference."
_, gw_t, gb_t = poisson_forward_backward(0.8, 0.5, np.array([1.0, -1.0]), np.array([2.0, 0.0]))
assert np.isclose(gb_t, np.mean(np.exp([1.3, -0.3]) - [2.0, 0.0])), "dL/db should be the mean of (lambda - y)."
print("Self-check passed.")

In [ ]:
# 2.3 (4 points): Gradients point uphill. Starting from w = 0.3, b = 0.1, take 200 steps of
#   w <- w - 0.1 * dL/dw,  b <- b - 0.1 * dL/db
# (Lesson 12 studies this rule) and store the final w_fit, b_fit and the loss history loss_hist (list of 201 values).

# YOUR CODE HERE
w_fit = b_fit = loss_hist = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.3
assert w_fit is not None and loss_hist is not None and len(loss_hist) == 201, "Fill in w_fit, b_fit and a loss history of 201 values."
assert loss_hist[-1] < loss_hist[0] and abs(w_fit - 0.8) < 0.2 and abs(b_fit - 0.5) < 0.2, "The fit should approach the generating parameters."
print("Self-check passed.")

---
## Exercise 3 — Extending an Automatic-Differentiation Engine (15 points)

The cell below contains the scalar `Value` class from the teaching notebook (addition, multiplication, powers and the sigmoid). Run it.

In [ ]:
class Value:
    def __init__(self, data, parents=(), op=''):
        self.data, self.grad = float(data), 0.0
        self._parents, self._op = parents, op
        self._backward = lambda: None
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad; other.grad += out.grad
        out._backward = _backward
        return out
    __radd__ = __add__
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad; other.grad += self.data * out.grad
        out._backward = _backward
        return out
    __rmul__ = __mul__
    def __sub__(self, other):
        return self + (other * -1.0)
    def __pow__(self, k):
        out = Value(self.data**k, (self,), f'**{k}')
        def _backward():
            self.grad += k * self.data**(k - 1) * out.grad
        out._backward = _backward
        return out
    def sigmoid(self):
        sv = 1 / (1 + np.exp(-self.data))
        out = Value(sv, (self,), 'sigmoid')
        def _backward():
            self.grad += sv * (1 - sv) * out.grad
        out._backward = _backward
        return out
    def backward(self):
        order, seen = [], set()
        def visit(v):
            if id(v) not in seen:
                seen.add(id(v))
                for p in v._parents:
                    visit(p)
                order.append(v)
        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()
print("Value class ready.")

In [ ]:
# 3.1 (8 points): Add three methods to Value, by assigning functions to the class (e.g. Value.tanh = my_tanh):
#   tanh(self)  -- output tanh(data), local derivative 1 - tanh^2
#   exp(self)   -- output e^data, local derivative e^data
#   relu(self)  -- output max(0, data), local derivative 1 if data > 0 else 0
# Each must create the output Value with self as parent and set out._backward with a += rule, as in sigmoid.

# YOUR CODE HERE
def v_tanh(self):
    raise NotImplementedError
def v_exp(self):
    raise NotImplementedError
def v_relu(self):
    raise NotImplementedError
Value.tanh, Value.exp, Value.relu = v_tanh, v_exp, v_relu

In [ ]:
# SELF-CHECK 3.1 -- compares your engine's gradients with central differences on an expression using all three.
def expr(a, b, c):
    return ((a * b).tanh() + (c * 0.5).exp() * a + (b - c).relu() * b)**2
def expr_float(a, b, c):
    return (np.tanh(a * b) + np.exp(c * 0.5) * a + max(0.0, b - c) * b)**2
vals = (0.7, -1.3, -2.1)
va, vb, vc = Value(vals[0]), Value(vals[1]), Value(vals[2])
out = expr(va, vb, vc); out.backward()
assert np.isclose(out.data, expr_float(*vals)), "The forward values of your new operations are wrong."
num = [central_diff(lambda t: expr_float(t, vals[1], vals[2]), vals[0]), central_diff(lambda t: expr_float(vals[0], t, vals[2]), vals[1]),
       central_diff(lambda t: expr_float(vals[0], vals[1], t), vals[2])]
assert np.allclose([va.grad, vb.grad, vc.grad], num, rtol=1e-6), "Backward rules disagree with central differences (check each += rule)."
print("Self-check passed.")

In [ ]:
# 3.2 (4 points): Use your engine to compute the gradient of the Poisson loss of Exercise 2 for ONE example
# (x = 1.5, y = 3, w = 0.3, b = 0.1): build L = lam - y * z with lam = (w*x + b).exp(), call backward(), and store
# auto_gw = w.grad, auto_gb = b.grad. Compare with the formula (lam - y) * x and (lam - y).

# YOUR CODE HERE
auto_gw = auto_gb = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.2
assert auto_gw is not None and auto_gb is not None, "auto_gw and auto_gb are still None."
lam_ = np.exp(0.55)
assert np.isclose(auto_gw, (lam_ - 3) * 1.5) and np.isclose(auto_gb, lam_ - 3), "The engine's gradients should match the formula."
print("Self-check passed.")

**3.3 (3 points, written response).** In 3.2, the node `z` is used twice (in `z.exp()` and in `z * 3.0`). Why must every backward rule use `+=` rather than `=`? What wrong answer would `=` give here?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 4 — Backpropagation Through an Affine Layer (15 points)

A layer maps a batch of inputs $\mathbf{A}$ (shape $(n, n_{\text{in}})$, one example per row) to $\mathbf{Z} = \mathbf{A}\mathbf{W}^{\mathsf T} + \mathbf{b}$ (shape $(n, n_{\text{out}})$). Given the upstream gradient $\boldsymbol\Delta = \partial L/\partial\mathbf{Z}$ (same shape as $\mathbf{Z}$), you need $\partial L/\partial\mathbf{W}$, $\partial L/\partial\mathbf{b}$ and $\partial L/\partial\mathbf{A}$.

In [ ]:
# 4.1 (7 points): Implement affine_backward(A, W, Delta) returning (dW, db, dA) with the right shapes:
#   dW = Delta^T A  (n_out, n_in),   db = column sums of Delta  (n_out,),   dA = Delta W  (n, n_in).
# Then check all three against central differences of the loss L = sum(G * Z), whose gradient with respect to Z is exactly G,
# for random A (6 x 4), W (3 x 4), b (3,) and G (6 x 3). Store the largest relative errors in aff_err (dict 'W', 'b', 'A').

def affine_backward(A, W, Delta):
    # YOUR CODE HERE
    raise NotImplementedError

A4, W4, b4, G4 = RNG.normal(size=(6, 4)), RNG.normal(size=(3, 4)), RNG.normal(size=3), RNG.normal(size=(6, 3))
aff_err = None

In [ ]:
# SELF-CHECK 4.1
assert aff_err is not None and set(aff_err) == {'W', 'b', 'A'}, "aff_err needs keys 'W', 'b', 'A'."
dW_, db_, dA_ = affine_backward(np.ones((5, 2)), np.ones((3, 2)), np.ones((5, 3)))
assert dW_.shape == (3, 2) and db_.shape == (3,) and dA_.shape == (5, 2), "Check the shapes of the three gradients."
assert max(aff_err.values()) < 1e-6, "At least one gradient disagrees with the central differences."
print("Self-check passed.")

In [ ]:
# 4.2 (4 points): A two-layer network with a tanh hidden layer: Z1 = X W1^T + b1, A1 = tanh(Z1), P = A1 W2^T + b2,
# loss L = 0.5 * mean over examples of ||P - Y||^2. Using affine_backward twice and the tanh slope, write
# two_layer_grads(X, Y, W1, b1, W2, b2) returning (dW1, db1, dW2, db2). Check dW1[0, 0] and dW2[0, 0] against
# central differences on the random problem below; store the relative errors in two_err (tuple of 2).

X4 = RNG.normal(size=(20, 5)); Y4 = RNG.normal(size=(20, 2))
W1_4, b1_4 = RNG.normal(0, 0.5, (8, 5)), RNG.normal(0, 0.1, 8)
W2_4, b2_4 = RNG.normal(0, 0.5, (2, 8)), RNG.normal(0, 0.1, 2)

def two_layer_grads(X, Y, W1, b1, W2, b2):
    # YOUR CODE HERE
    raise NotImplementedError

two_err = None

In [ ]:
# SELF-CHECK 4.2
assert two_err is not None and len(two_err) == 2, "two_err should hold two relative errors."
assert max(two_err) < 1e-6, "two_layer_grads disagrees with the central differences."
print("Self-check passed.")

**4.3 (4 points, written response).** Explain why $\partial L/\partial\mathbf{W}$ is an outer product summed over the batch, and why the gradient passed to the layer below is $\boldsymbol\Delta\mathbf{W}$ (the forward matrix used "backwards"). Why would exact backpropagation in the brain require feedback connections whose strengths mirror the forward synapses, and what did feedback-alignment experiments show?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — Vanishing and Exploding Gradients Through Time (15 points)

A single recurrent neuron obeys $x_{t+1} = \tanh(w\,x_t + u_t)$ with a small input $u_t$. How much does the state at time $T$ depend on the state at time 0? By the chain rule

$$\frac{\partial x_T}{\partial x_0} = \prod_{t=0}^{T-1}\frac{\partial x_{t+1}}{\partial x_t} = \prod_{t=0}^{T-1} w\,\big(1 - x_{t+1}^2\big).$$

In [ ]:
# 5.1 (6 points): Write sensitivity_through_time(w, T, x0=0.1, u=None) that simulates the neuron for T steps (u: array of
# T inputs, default zeros) and returns |dx_T/dx_0| computed with the product formula. Check it against a central difference
# (perturb x0) for w = 0.9, T = 20. Then compute sens5[w] for w in w_list and T = 50, with u5 below.

w_list = [0.5, 0.9, 1.0, 1.1, 1.5, 3.0]
u5 = RNG.normal(0, 0.1, 50)

# YOUR CODE HERE
def sensitivity_through_time(w, T, x0=0.1, u=None):
    raise NotImplementedError
sens5 = None

In [ ]:
# SELF-CHECK 5.1
assert sens5 is not None and set(sens5) == set(w_list), "sens5 needs one value per w."
assert np.isclose(sensitivity_through_time(0.9, 20), abs(central_diff(lambda x0: simulate(0.9, 20, x0, np.zeros(20)), 0.1)), rtol=1e-5), "Product formula disagrees with the simulation."
assert sens5[0.5] < 1e-10, "With w = 0.5 the sensitivity should vanish."
print("Self-check passed.")

In [ ]:
# 5.2 (5 points): Plot |dx_T/dx_0| against T = 1 ... 100 (log y-axis) for w = 0.9, 1.0, 1.1, 1.5 and 3.0, with u = zeros
# and x0 = 0.1. Store the curves in curves5 (dict w -> array of 100 values) and, for each w, the final state x_100 in
# xfinal5 (dict). (You may need a separate small simulation to get x_100.)

# YOUR CODE HERE
curves5 = xfinal5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.2
assert curves5 is not None and xfinal5 is not None, "curves5 and xfinal5 are still None."
assert all(np.shape(v) == (100,) for v in curves5.values()), "Each curve should have 100 values."
assert curves5[0.9][-1] < 1e-3 and curves5[3.0][-1] < 1e-3, "Both small w and large (saturating) w should give vanishing sensitivity."
print("Self-check passed.")

**5.3 (4 points, written response).** Explain your curves. Why does $w = 0.9$ vanish? Why does $w = 3$ also vanish, although $|w| > 1$ (look at $x_{100}$ and the factor $1 - x^2$)? What happens near $w = 1$, and how does this connect to Lesson 6's spectral radius? Why does it make learning long-range dependencies hard?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 6 — Input Sensitivity of a Network (15 points)

A trained network's derivative with respect to its **inputs** says which inputs it is sensitive to, and lets it be approximated locally by a linear map. The network below has fixed weights: 3 inputs, 16 tanh units, 1 output.

In [ ]:
rng6 = np.random.default_rng(6)
W1_6, b1_6 = rng6.normal(0, 1.0, (16, 3)), rng6.normal(0, 0.5, 16)
w2_6, b2_6 = rng6.normal(0, 0.5, 16), 0.2
def net6(x):
    # x: shape (3,) or (n, 3). Returns the scalar output (or n outputs).
    return np.tanh(np.atleast_2d(x) @ W1_6.T + b1_6) @ w2_6 + b2_6
print("net6 ready; output at the origin:", net6(np.zeros(3)))

In [ ]:
# 6.1 (6 points): Write input_grad6(x) returning the gradient of net6 with respect to its 3 inputs at a single point x
# (shape (3,)), by backpropagation: d out/d A1 = w2, d A1/d Z1 = 1 - A1^2, d Z1/d x = W1. Check it against central
# differences in each input direction at x = (0.3, -0.5, 1.0), storing the largest absolute difference in grad6_err.

# YOUR CODE HERE
def input_grad6(x):
    raise NotImplementedError
grad6_err = None

In [ ]:
# SELF-CHECK 6.1
assert grad6_err is not None, "grad6_err is still None."
assert grad6_err < 1e-8, "input_grad6 disagrees with the central differences."
print("Self-check passed.")

In [ ]:
# 6.2 (5 points): Local linear approximation of the network. At x6, predict net6(x6 + d) by net6(x6) + input_grad6(x6) . d
# for random directions d of length eps in eps_list = [0.01, 0.1, 0.5, 1.0, 2.0] (average the absolute error over 200
# random unit directions times eps, using rng = np.random.default_rng(1)). Store the mean errors in lin_err6 (array of 5)
# and plot them against eps on log-log axes.

eps_list = [0.01, 0.1, 0.5, 1.0, 2.0]

# YOUR CODE HERE
lin_err6 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.2
assert lin_err6 is not None and np.shape(lin_err6) == (5,), "lin_err6 should hold five values."
assert lin_err6[0] < 1e-3 and np.all(np.diff(lin_err6) > 0), "The error should be tiny for small steps and grow with the step."
print("Self-check passed.")

**6.3 (4 points, written response).** Why does the error of the linear approximation grow roughly like the square of the step at first? Your teaching notebook's mini-lab computed $\partial\hat v_x/\partial(\text{rate})$ for a decoding network in every time bin. What does the average of these input gradients tell you, what does their variation tell you, and what can they **not** tell you?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: Backpropagation for a Deeper Decoder (15 points)

In the teaching notebook a one-hidden-layer tanh network was compared with a linear decoder of wheel velocity. Here you write backpropagation for a **two-hidden-layer** network (437 → 32 → 16 → 1) for any activation, check it on the mouse motor-cortex data ([Steinmetz et al., 2019](https://doi.org/10.1038/s41586-019-1787-x); Neuromatch Academy version; the original data are in the public domain, CC0), and compare tanh with ReLU.

The cell below prepares the data exactly as in the teaching notebook (1.5 MB, downloaded the first time), fits the linear decoder, and provides `init_params` and a training loop that calls **your** functions. Do not modify it.

In [ ]:
# PROVIDED -- do not modify.
import os, urllib.request
from scipy.ndimage import gaussian_filter1d

MOP_FILE = 'steinmetz_s10_hench_mop.npz'
for candidate in [MOP_FILE, os.path.join('..', '..', 'data', MOP_FILE)]:
    if os.path.exists(candidate):
        mop_path = candidate
        break
else:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/peter-adepoju/math-neuro-ml/51d88220064023b5d3240c39e19b07acc7896f44/data/' + MOP_FILE, MOP_FILE)
    mop_path = MOP_FILE
mop = np.load(mop_path)
n_tr7, n_neu7, _ = mop['spikes'].shape
rates7 = mop['spikes'].reshape(n_tr7, n_neu7, 50, 5).sum(axis=3) / 0.05                   # (342, 447, 50) spikes/s
X_raw = gaussian_filter1d(rates7, 1, axis=2).transpose(0, 2, 1).reshape(-1, n_neu7)       # (17100, 447): rows = (trial, bin)
Y_raw = (mop['wheel'].reshape(n_tr7, 50, 5).sum(axis=2) / 0.05).reshape(-1, 1)            # (17100, 1): wheel velocity, steps/s
train7 = np.repeat(np.arange(n_tr7) < n_tr7 // 2, 50)
X_raw = X_raw[:, X_raw[train7].std(axis=0) > 0]
Xs = (X_raw - X_raw[train7].mean(0)) / X_raw[train7].std(0)               # (17100, 437) standardised rates
Ys = (Y_raw - Y_raw[train7].mean(0)) / Y_raw[train7].std(0)               # (17100, 1) standardised velocity
Xtr7, Ytr7, Xte7, Yte7 = Xs[train7], Ys[train7], Xs[~train7], Ys[~train7]
SIZES7 = [Xs.shape[1], 32, 16, 1]

def r2(P, Yt):
    return float(1 - np.sum((Yt - P)**2) / np.sum((Yt - Yt.mean(axis=0))**2))
Dlin = np.column_stack([np.ones(len(Ys)), Xs])
B7 = np.linalg.lstsq(Dlin[train7], Ytr7, rcond=None)[0]
r2_lin7 = r2(Dlin[~train7] @ B7, Yte7)

def init_params(sizes, act, seed=0):
    rng = np.random.default_rng(seed)
    gain = 2.0 if act == 'relu' else 1.0
    return [(rng.normal(0, np.sqrt(gain / sizes[k]), (sizes[k + 1], sizes[k])), np.zeros(sizes[k + 1])) for k in range(len(sizes) - 1)]

def train(params, act, steps=400, eta=0.05):
    # Plain gradient descent using YOUR forward7 and backward7.
    for _ in range(steps):
        cache = forward7(params, Xtr7, act)
        grads = backward7(params, cache, Ytr7, act)
        params = [(W - eta * gW, b - eta * gb) for (W, b), (gW, gb) in zip(params, grads)]
    return params

print("data:", Xs.shape, Ys.shape, "  network sizes:", SIZES7, "  linear decoder test R^2:", round(r2_lin7, 3))

In [ ]:
# 7.1 (6 points): Write
#   forward7(params, X, act) -> cache, a list [A0 = X, (Z1, A1), (Z2, A2), (Z3, P)] where hidden layers use the activation
#                               act ('tanh' or 'relu') and the last layer is linear (P = Z3);
#   backward7(params, cache, Y, act) -> [(dW1, db1), (dW2, db2), (dW3, db3)] for L = 0.5 * mean over rows of ||P - Y||^2.
# Then gradient-check BOTH activations on the first 200 training rows (params = init_params(SIZES7, act)) for
# 20 randomly chosen weights spread over the three layers (rng = np.random.default_rng(3)); store the largest relative
# errors in gcheck7 = {'tanh': ..., 'relu': ...}.

# YOUR CODE HERE
def forward7(params, X, act):
    raise NotImplementedError
def backward7(params, cache, Y, act):
    raise NotImplementedError
gcheck7 = None

In [ ]:
# SELF-CHECK 7.1
assert gcheck7 is not None and set(gcheck7) == {'tanh', 'relu'}, "gcheck7 needs 'tanh' and 'relu'."
assert gcheck7['tanh'] < 1e-6, "The tanh network's gradients disagree with central differences."
assert gcheck7['relu'] < 1e-4, "The ReLU network's gradients disagree with central differences (allowing for an occasional kink)."
c_ = forward7(init_params(SIZES7, 'relu'), Xtr7[:5], 'relu')
assert len(c_) == 4 and c_[-1][1].shape == (5, 1), "forward7 should return [X, (Z1, A1), (Z2, A2), (Z3, P)]."
print("Self-check passed.")

In [ ]:
# 7.2 (5 points): Train both networks with the provided train() (400 steps, eta = 0.05, init_params seed 0) and store
# the test R^2 as r2_net7 = {'tanh': ..., 'relu': ...} (one number each), and the training R^2 as r2_train7 likewise.
# Also store dead7: the fraction of first-hidden-layer ReLU units that are zero for EVERY test row after training.
# Print a small table together with the linear decoder (r2_lin7).

# YOUR CODE HERE
r2_net7 = r2_train7 = dead7 = None
trained7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.2
assert r2_net7 is not None and r2_train7 is not None and dead7 is not None, "Fill in r2_net7, r2_train7 and dead7."
assert set(r2_net7) == {'tanh', 'relu'} and all(np.ndim(v) == 0 for v in r2_net7.values()), "One test R^2 per network."
assert all(r2_train7[a] > r2_net7[a] for a in r2_net7), "Training R^2 should exceed test R^2."
print("Self-check passed.")

In [ ]:
# 7.3 (2 points): Backpropagate to the inputs of the trained ReLU network: for every test row compute d(v_hat)/dx
# (one value per input neuron) and store the mean over rows in sens7. Store its correlation with the linear decoder's
# weights (B7[1:, 0]) in corr7.

# YOUR CODE HERE
sens7 = corr7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.3
assert sens7 is not None and corr7 is not None and np.shape(sens7) == (Xs.shape[1],), "sens7 should have one value per input neuron."
assert -1 <= corr7 <= 1, "corr7 is a correlation."
print("Self-check passed.")

**7.4 (2 points, written response — interpretation and limitation).** Compare the three decoders. Does either network beat the linear decoder? Suggest, using this lesson, why ReLU and tanh might train differently in the same number of steps. What does `corr7` say about how the ReLU network uses the neurons? Give one limitation of the comparison.

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — Forward-Mode Differentiation (extra credit, not part of the 100 points)

Backpropagation is **reverse mode**: one backward pass gives the derivative of **one** output with respect to **all** inputs. **Forward mode** carries a derivative *forwards* alongside each value: every number $a$ is paired with its derivative $\dot a = da/dx$ with respect to one chosen input, and each operation updates both (for a product, $\dot{(ab)} = \dot a b + a\dot b$).

1. Write a tiny `Dual` class (value, derivative) supporting `+`, `*` and `tanh`, and use it to compute $\frac{d}{dx}\tanh(3x\cdot x + x)$ at $x = 0.4$. Check against a central difference.
2. Explain when forward mode is cheaper than reverse mode (think: number of inputs versus number of outputs), and why training neural networks always uses reverse mode.

**Your code:**

In [ ]:
# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, and a legend where appropriate.
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.